# Bản cuối (run G): `v7_s4_p3` train trên toàn bộ 6.000 ảnh

Cấu hình giống hệt `v7_s4_p3` (val float 0,9793): head stride 4 (`EDGE_STRIDES=4,8,16`), `--arch 64,64,72`, 198.196 tham số,
120 epoch, batch 16, lr 2e-3, seed 42, không teacher. Khác duy nhất: `--val-frac 0` (thêm 600 ảnh val vào train).
Không còn tập val nên không có mAP; mô hình dùng là bản EMA ở epoch cuối (`last.keras`), như run C (bản tốt nhất của các run
có val đều ở sát cuối).

**Trước khi chạy:** Add Input dữ liệu cuộc thi + Dataset chứa `edgeai_viec7.zip` (dùng lại dataset của Việc 7);
Settings: **GPU T4 x2**, **Internet On**. Chạy bằng **Save Version → Save & Run All (Commit)**. Khoảng 3,5 giờ.
Kết quả: `g_s4_p3.zip` ở tab **Output** → giải nén vào `runs/g_s4_p3/` ở máy → chạy cell **Bản cuối** trong `notebooks/train_det.ipynb`.

## 1. Cài Keras 3.15.1 và kiểm GPU

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras==3.15.1"], check=True)
chk = ("import os; os.environ['KERAS_BACKEND']='torch'; import keras, torch; "
       "print('keras', keras.__version__, '| torch', torch.__version__); "
       "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG THAY GPU'); "
       "assert keras.__version__ == '3.15.1' and torch.cuda.is_available()")
subprocess.run([sys.executable, "-c", chk], check=True)

## 2. Tìm dữ liệu và code

In [ ]:
import json, os, shutil, time, zipfile
from pathlib import Path

INP, WORK = Path("/kaggle/input"), Path("/kaggle/working")


def find(test, what, base=INP, depth=5):
    for k in range(1, depth + 1):
        for d in sorted(base.glob("/".join(["*"] * k))):
            if test(d):
                return d
    raise FileNotFoundError(f"Không thấy {what} trong {base}: kiểm tra đã Add Input chưa")


DATA = find(lambda d: (d / "train" / "images").is_dir() and (d / "train" / "labels").is_dir(), "dữ liệu cuộc thi")
try:
    PKG = find(lambda d: (d / "pipeline" / "det" / "model.py").is_file()
               and "base_for" in (d / "pipeline" / "det" / "model.py").read_text(encoding="utf-8"), "gói edgeai_viec7")
except FileNotFoundError:
    z = find(lambda d: d.is_file() and d.name == "edgeai_viec7.zip", "edgeai_viec7.zip")
    PKG = WORK / "pkg"
    zipfile.ZipFile(z).extractall(PKG)
PIPE = WORK / "pipeline"
shutil.copytree(PKG / "pipeline", PIPE, dirs_exist_ok=True)
n_img = len(list((DATA / "train" / "images").glob("*.png")))
assert n_img == 6000, f"{n_img} ảnh train, cần 6000"
print("DATA:", DATA, f"({n_img} ảnh) | PKG:", PKG, "| commit", (PIPE / "GIT_COMMIT").read_text().strip())

## 3. Train trên toàn bộ dữ liệu (khoảng 3,5 giờ)

In [ ]:
NAME = "g_s4_p3"
OUT = WORK / "runs" / NAME
ENV = {**os.environ, "KERAS_BACKEND": "torch", "EDGE_IMGSZ": "320x240", "EDGE_STRIDES": "4,8,16",
       "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "CUDA_VISIBLE_DEVICES": "0"}
cmd = [sys.executable, str(PIPE / "det/train_det.py"), "--data", str(DATA), "--out", str(OUT), "--epochs", "120",
       "--batch", "16", "--eval-every", "5", "--val-frac", "0", "--arch", "64,64,72", "--no-export"]
print(">", " ".join(cmd), flush=True)
t0, log = time.time(), []
p = subprocess.Popen(cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    log.append(line)
    print(line, end="", flush=True)
code = p.wait()
print(f"\nXong sau {(time.time() - t0) / 60:.1f} phút, mã thoát {code}")
assert code == 0, "Lệnh lỗi, xem log phía trên"
text = "".join(log)
assert "stride: 4,8,16" in text and "198,196" in text, "Sai cấu hình (không thấy stride 4,8,16 / 198,196 tham số)"
assert json.load(open(OUT / "split.json"))["val_ids"] == [], "Tập val phải rỗng (--val-frac 0)"

## 4. Đóng gói kết quả

In [ ]:
Z = WORK / f"{NAME}.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_STORED) as z:
    for f in ("last.keras", "log.json", "split.json", "train_meta.json"):
        z.write(OUT / f, f)
for r in json.load(open(OUT / "log.json"))[-4:]:
    print(r)
for d in (PIPE, WORK / "pkg", WORK / "runs"):
    shutil.rmtree(d, ignore_errors=True)
print(f"Đã tạo {Z} ({Z.stat().st_size / 2**20:.1f} MB)")